# ALSApp - Plant disease classifier (on-device)

Trains a small MobileNetV2-based classifier and exports it as a `.tflite` file small
enough to ship inside the Flutter app and run fully offline on the phone.

**Training data**
- **PlantVillage** - 38 classes across 14 crops, ~54k lab-style leaf photos. Bundled in
  TensorFlow Datasets, no auth needed.
- **Wheat** - field photos from the John Innes Centre (yellow rust, brown rust, septoria,
  mildew, healthy). Kaggle `yasserhessein/wheat-disease-dataset-small`
  (also on Zenodo, record 7307816). PlantVillage has no wheat at all.
- **Olive** - orchard photos from Denizli, Turkey (Aculus olearius, peacock spot, healthy).
  Kaggle `habibulbasher01644/olive-leaf-image-dataset`. Not in PlantVillage either.

Labels are `<Crop>___<Condition>`, e.g. `Wheat___Brown_rust`. The app uses the part before
`___` to limit the answer to the crop the farmer says the photo shows, and it only offers
photo diagnosis for crops whose prefix appears in `plant_labels.txt` - so after retraining
you just replace the two asset files, no app code change.

**Before running**
1. `Runtime` -> `Change runtime type` -> select a `T4 GPU`.
2. The wheat and olive data come from Kaggle, so you need a free API token (same as the
   cattle notebook): kaggle.com -> your avatar -> `Settings` -> `API` -> `Create New Token`.
   In this Colab, click the key icon (🔑) in the left sidebar ("Secrets") and add two secrets,
   `KAGGLE_USERNAME` and `KAGGLE_KEY`, with notebook access enabled for both.

**At the end** this notebook downloads two files - save them into
`frontend/assets/models/` in the app repo, replacing the old ones:
- `plant_disease.tflite`
- `plant_labels.txt`

**Read the per-crop accuracy printout before shipping** - the wheat and olive lines are the
ones that matter for this app's farmers.

In [ ]:
!pip install -q -U tensorflow tensorflow-datasets scikit-learn kagglehub

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import classification_report

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

# plant_village only ships a single 'train' split - carve out a held-out
# slice ourselves so we have an honest accuracy number at the end.
(train_raw, val_raw), ds_info = tfds.load(
    'plant_village',
    split=['train[:90%]', 'train[90%:]'],
    as_supervised=True,
    with_info=True,
)

class_names = ds_info.features['label'].names
num_classes = ds_info.features['label'].num_classes
print(f"{num_classes} classes, e.g.: {class_names[:5]}")

## Wheat and olive (field photos)

PlantVillage is above. The next cell adds the crops it lacks. Each Kaggle dataset's class
folders are matched to a label by keyword on the *folder name* (`EXTRA_CROPS` below). Kaggle
layouts aren't machine-readable, so the cell prints exactly what it found - **check that
table**. If a folder can't be matched it stops with the folder list rather than silently
dropping a class; fix it by editing `rules` (or `ignore`) and re-running.

In [ ]:
import os, hashlib
from pathlib import Path
import kagglehub
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Label = '<Crop>___<suffix>'. `rules`: (keyword in lowercase folder name, label suffix),
# first match wins. `ignore`: folder-name keywords to skip.
EXTRA_CROPS = {
    'Wheat': {
        'kaggle': 'yasserhessein/wheat-disease-dataset-small',
        'rules': [('yellow', 'Yellow_rust'), ('brown', 'Brown_rust'), ('septoria', 'Septoria'),
                  ('mildew', 'Mildew'), ('health', 'healthy')],
        'ignore': [],
    },
    'Olive': {
        'kaggle': 'habibulbasher01644/olive-leaf-image-dataset',
        'rules': [('aculus', 'Aculus_olearius'), ('mite', 'Aculus_olearius'),
                  ('peacock', 'Peacock_spot'), ('spot', 'Peacock_spot'), ('health', 'healthy')],
        'ignore': [],
    },
}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
VAL_FRACTION = 0.15  # only used when a dataset doesn't ship its own train/test split


def list_class_folders(root):
    """{directory: [image paths]} for every directory that directly holds images."""
    found = {}
    for dirpath, _, filenames in os.walk(root):
        imgs = sorted(os.path.join(dirpath, f) for f in filenames
                      if os.path.splitext(f)[1].lower() in IMAGE_EXTS)
        if imgs:
            found[dirpath] = imgs
    return found


def split_of(path):
    """'train' / 'val' if the dataset already ships a split (we keep it, so near-duplicate
    images can't leak across it), else None."""
    parts = {p.lower() for p in Path(path).parts}
    if parts & {'test', 'val', 'valid', 'validation'}:
        return 'val'
    if parts & {'train', 'training'}:
        return 'train'
    return None


def is_val(path, root):
    """Deterministic 85/15 split by file path, for images with no shipped split."""
    rel = os.path.relpath(path, root)
    return int(hashlib.md5(rel.encode()).hexdigest(), 16) % 100 < VAL_FRACTION * 100


def collect_crop(crop, spec, root):
    """-> {label_suffix: {'train': [paths], 'val': [paths]}}. Raises on unmappable folders."""
    labelled, unmapped = {}, []
    for folder, imgs in list_class_folders(root).items():
        name = os.path.basename(folder).lower()
        if any(k in name for k in spec['ignore']):
            continue
        suffix = next((s for kw, s in spec['rules'] if kw in name), None)
        if suffix is None:
            unmapped.append(f"{os.path.relpath(folder, root)}  ({len(imgs)} images)")
            continue
        bucket = labelled.setdefault(suffix, {'train': [], 'val': []})
        forced = split_of(os.path.relpath(folder, root))
        for p in imgs:
            side = forced or ('val' if is_val(p, root) else 'train')
            bucket[side].append(p)
    if unmapped:
        raise ValueError(
            f"{crop} ({spec['kaggle']}): can't map these folders to a label:\n  "
            + "\n  ".join(unmapped)
            + "\nAdd a keyword to EXTRA_CROPS[...]['rules'] (or 'ignore') and re-run this cell.")
    if len(labelled) < 2:
        raise ValueError(f"{crop}: found only {sorted(labelled)} - expected several classes. "
                         f"Layout under {root}:\n" + "\n".join(list(list_class_folders(root))[:20]))
    return labelled


extra_labels = []     # names appended after PlantVillage's 38, in order
extra_files = {}      # crop -> {'train': [(path, global_label)], 'val': [...]}
for crop, spec in EXTRA_CROPS.items():
    root = kagglehub.dataset_download(spec['kaggle'])
    labelled = collect_crop(crop, spec, root)
    extra_files[crop] = {'train': [], 'val': []}
    for suffix in sorted(labelled):
        name = f"{crop}___{suffix}"
        idx = len(class_names) + len(extra_labels)
        extra_labels.append(name)
        for side in ('train', 'val'):
            extra_files[crop][side] += [(p, idx) for p in labelled[suffix][side]]
        print(f"  {name:32s} label {idx:2d}   train {len(labelled[suffix]['train']):5d}   "
              f"val {len(labelled[suffix]['val']):4d}")

class_names_all = list(class_names) + extra_labels
num_classes = len(class_names_all)
print(f"\n{num_classes} classes total ({len(class_names)} PlantVillage + {len(extra_labels)} new)")

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

def prep(image, label, training):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    if training:
        image = augment(image, training=True)
    return image, label

# --- wheat / olive: decode once, to the same 256x256 uint8 PlantVillage uses, so the
# three sources can be mixed in one pipeline (and cached in RAM: field photos are big).
def load_extra(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (256, 256))
    return tf.cast(tf.round(img), tf.uint8), label

def extra_ds(files):
    paths = [p for p, _ in files]
    labels = np.array([l for _, l in files], dtype=np.int64)
    return (tf.data.Dataset.from_tensor_slices((paths, labels))
            .map(load_extra, num_parallel_calls=tf.data.AUTOTUNE)
            .cache())

# PlantVillage has ~49k training photos, wheat only ~1k. Repeat the small crops (each pass
# gets different random augmentation) so every crop is seen a meaningful amount per epoch.
TARGET_PER_CROP = 5000

n_pv_train = int(train_raw.cardinality())
if n_pv_train < 0:  # cardinality unknown
    n_pv_train = int(ds_info.splits['train'].num_examples * 0.9)

train_parts, train_sizes = [train_raw], [n_pv_train]
val_parts = [val_raw]
for crop, files in extra_files.items():
    n = len(files['train'])
    repeat = max(1, round(TARGET_PER_CROP / n))
    train_parts.append(extra_ds(files['train']).repeat(repeat))
    train_sizes.append(n * repeat)
    val_parts.append(extra_ds(files['val']))
    print(f"{crop}: {n} train photos x{repeat} per epoch, {len(files['val'])} held out")

weights = [s / sum(train_sizes) for s in train_sizes]
train_mixed = tf.data.Dataset.sample_from_datasets(train_parts, weights=weights, seed=42)
# sample_from_datasets forgets how many elements it yields, which makes Keras show an unknown
# step count. Every source is finite and all get fully consumed, so the total is exactly the
# sum - restore it (TF also checks it while training).
if int(train_raw.cardinality()) >= 0:
    train_mixed = train_mixed.apply(tf.data.experimental.assert_cardinality(sum(train_sizes)))
val_mixed = val_parts[0]
for part in val_parts[1:]:
    val_mixed = val_mixed.concatenate(part)

train_ds = (train_mixed
            .map(lambda x, y: prep(x, y, True), num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(2000)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))
val_ds = (val_mixed
          .map(lambda x, y: prep(x, y, False), num_parallel_calls=tf.data.AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

In [ ]:
# Rescaling is baked into the model itself (not done in the tf.data pipeline)
# so the Flutter side can feed plain 0-255 RGB pixels with no normalization math.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.Rescaling(1. / 127.5, offset=-1)(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Phase 1: train just the new head, backbone frozen.
model.fit(train_ds, validation_data=val_ds, epochs=6)

In [ ]:
# Phase 2: unfreeze the top of the backbone and fine-tune a few epochs at a
# low learning rate - usually a solid accuracy bump for not much extra time.
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=4)

In [ ]:
# Honest held-out numbers before we ship this anywhere.
y_true, all_probs = [], []
for images, labels in val_ds:
    all_probs.append(model.predict(images, verbose=0))
    y_true.extend(labels.numpy())
y_true = np.array(y_true)
probs = np.concatenate(all_probs)
y_pred = probs.argmax(axis=1)

print(classification_report(y_true, y_pred, labels=list(range(num_classes)),
                            target_names=class_names_all, zero_division=0))

### Per-crop accuracy - what the farmer will actually experience

The app already knows the crop, so it only lets the model choose among that crop's own
labels. This scores it the same way. **Wheat and olive are the lines to check**: if either
is well below ~80% (the wheat set is small), don't ship this model yet - more data or a
second wheat dataset is needed. Remember these are photos from the same sources as the
training data; phone photos in a real field will do somewhat worse.

In [ ]:
crops = sorted({n.split('___')[0] for n in class_names_all})
for crop in crops:
    idx = np.array([i for i, n in enumerate(class_names_all) if n.startswith(crop + '___')])
    mask = np.isin(y_true, idx)
    if not mask.any():
        continue
    within = idx[probs[mask][:, idx].argmax(axis=1)]
    acc = (within == y_true[mask]).mean()
    flag = '  <-- new crop, check this' if crop in EXTRA_CROPS else ''
    print(f"{crop:28s} {acc * 100:5.1f}%   ({int(mask.sum()):5d} held-out photos, {len(idx):2d} classes){flag}")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# NOTE: no converter.optimizations (dynamic-range quantization) on purpose. TF >= 2.13 then emits
# FULLY_CONNECTED v12, which the TFLite runtimes bundled by tflite_flutter on macOS (2.11) and
# iOS (2.12) cannot load ("Didn't find op for builtin opcode FULLY_CONNECTED version 12").
# Plain float32 is ~9 MB and runs on every platform.
tflite_model = converter.convert()

with open('plant_disease.tflite', 'wb') as f:
    f.write(tflite_model)
with open('plant_labels.txt', 'w') as f:
    f.write('\n'.join(class_names_all))

import os
print(f"plant_disease.tflite: {os.path.getsize('plant_disease.tflite') / 1e6:.1f} MB, "
      f"{len(class_names_all)} labels, crops: {sorted({n.split('___')[0] for n in class_names_all})}")

In [ ]:
from google.colab import files
files.download('plant_disease.tflite')
files.download('plant_labels.txt')